# 06 · Validação e Sanidade do Score

**Objetivo:** checar estabilidade temporal, calibração agregada e monotonicidades.
Não há rótulo individual → validação **agregada** (consistência interna), não acurácia individual.

**Entradas:** `data/interim/val_<ano>.parquet` (agregados por ano do nb03).  
**Saídas:** `outputs/tables/validacao*.csv`, figuras.

**Limitações:** sem *ground truth* individual; confirma estabilidade/consistência, não acurácia.

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
import numpy as np
from src import cells, rates, viz
interim = cfg['abs']['interim']
motivos = cfg['motivos']
VAL_LVL = 'cbo2_cnae2_tempo_uf'
cols = [l['cols'] for l in cells.BACKOFF_LEVELS if l['name']==VAL_LVL][0]
anos = sorted(int(f.stem.split('_')[1]) for f in interim.glob('val_*.parquet'))
assert len(anos) >= 2, 'Validação temporal requer >=2 anos (ajuste cfg.anos).'
treino_anos, holdout_ano = anos[:-1], anos[-1]
print('Treino:', treino_anos, '| Holdout:', holdout_ano)

In [ ]:
# Estabilidade temporal: taxa (treino) vs taxa observada (holdout) por célula,
# a partir dos agregados por ano (nível populoso VAL_LVL).
load = lambda a: pd.read_parquet(interim / f'val_{a}.parquet')
tr = None
for a in treino_anos:
    tr = rates._sum_two_tables(tr, load(a), cols, motivos)
ho = load(holdout_ano)
def add_taxa(df):
    df = df.copy(); df['taxa'] = df['k_involuntario_sjc'] / df['n']; return df
tr, ho = add_taxa(tr), add_taxa(ho)
merged = tr.merge(ho, on=cols, suffixes=('_tr','_ho'))
merged = merged[merged['n_tr'] >= cfg['suavizacao']['exposicao_minima']]
corr = merged['taxa_tr'].corr(merged['taxa_ho'])
mae = (merged['taxa_tr']-merged['taxa_ho']).abs().mean()
print(f'Estabilidade célula a célula: corr={corr:.3f}  MAE={mae:.4f}  (n_células={len(merged):,})')

In [ ]:
# Calibração agregada: agrupa células por decil de taxa prevista (treino)
merged['decil'] = pd.qcut(merged['taxa_tr'], 10, duplicates='drop')
calib = merged.groupby('decil', observed=True).apply(
    lambda d: pd.Series({'prevista': np.average(d['taxa_tr'], weights=d['n_tr']),
                         'observada': np.average(d['taxa_ho'], weights=d['n_ho'])}))
calib.to_csv(cfg['abs']['tables'] / 'validacao_calibracao.csv')
fig, ax = __import__('matplotlib.pyplot', fromlist=['x']).subplots(figsize=(5,5))
ax.plot([0, calib['prevista'].max()], [0, calib['prevista'].max()], '--', color='gray')
ax.scatter(calib['prevista'], calib['observada'])
ax.set_xlabel('Taxa prevista (treino)'); ax.set_ylabel('Taxa observada (holdout)')
ax.set_title('Calibração agregada por decil'); viz.save_fig(fig, cfg['abs']['figures']/'calibracao.png'); fig

In [ ]:
# Sanidade de monotonicidade: risco deve cair com o tempo de vínculo
from src import scoring
base = dict(cbo='521110', cnae='4711301', uf='PA', idade=40,
            escolaridade='medio_completo', tamanho_empresa=1)
riscos = [scoring.score_pessoa(**base, tempo_vinculo_meses=t)['risco']['involuntario_sjc'][12]
          for t in [2, 8, 18, 72]]
print('Riscos 12m por tempo [2,8,18,72]m:', [round(x,3) for x in riscos])
print('Monotonicamente não-crescente?', all(a>=b-1e-9 for a,b in zip(riscos, riscos[1:])))

In [ ]:
# Sensibilidade ao shrinkage (m) — reusa UM único Scorer (memória),
# variando m diretamente em rates.eb_annual_hazard.
from src.scoring import Scorer
sc = Scorer(cfg=cfg)
keys = sc._keys_from_attrs(cbo='223505', cnae='8610101', uf='AM', idade=62,
                           escolaridade='superior', tempo_vinculo_meses=5, tamanho_empresa=0)
mot = 'involuntario_sjc'
for m in [10, 50, 200]:
    haz, nivel, n = rates.eb_annual_hazard(sc.indexes, sc.meta, keys, mot, m)
    print(f'm={m:>3} -> risco12m={rates.horizon_risk(haz,12):.3f} nível={nivel} exp={n:,}')